# FaceForensics++ — Extract Face Crops (Kaggle)

**Dataset:** `/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/`  
**Output:** `/kaggle/working/extracted_faces/` → tải về máy hoặc lưu thành Kaggle Dataset

**Pipeline:**  
1. Cài thư viện  
2. Tạo split manifest 72/14/14  
3. Extract 32 frame/video (uniform) bằng MTCNN  
4. Lưu thành Kaggle Dataset để dùng lại khi training

In [1]:
import torch, os
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Kiem tra dataset
DATASET_ROOT = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
for cls in sorted(os.listdir(DATASET_ROOT)):
    p = os.path.join(DATASET_ROOT, cls)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f"  {cls:25s}: {n} files")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
  DeepFakeDetection        : 1000 files
  Deepfakes                : 1000 files
  Face2Face                : 1000 files
  FaceShifter              : 1000 files
  FaceSwap                 : 1000 files
  NeuralTextures           : 1000 files
  csv                      : 10 files
  original                 : 1000 files


In [2]:
!pip install -q facenet-pytorch
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.

In [3]:
from facenet_pytorch import MTCNN
import torch
print("facenet-pytorch OK")
print(f"CUDA: {torch.cuda.is_available()}")

RuntimeError: operator torchvision::nms does not exist

In [ ]:
import os

DATASET_ROOT     = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
CSV_DIR          = os.path.join(DATASET_ROOT, "csv")
EXTRACTED_ROOT   = "/kaggle/working/extracted_faces"
MANIFEST_PATH    = "/kaggle/working/splits/manifest.json"
FRAMES_PER_VIDEO = 32
FAKE_CLASSES     = ["DeepFakeDetection", "Deepfakes", "Face2Face",
                    "FaceShifter", "FaceSwap", "NeuralTextures"]
REAL_CLASS       = "original"

os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
os.makedirs(EXTRACTED_ROOT, exist_ok=True)

print(f"DATASET  : {DATASET_ROOT}")
print(f"OUTPUT   : {EXTRACTED_ROOT}")
print(f"MANIFEST : {MANIFEST_PATH}")

## Bước 1 — Tạo Split Manifest (72/14/14)

In [ ]:
import json, random, pandas as pd
from pathlib import Path

def _read_csv(csv_path):
    try:
        df = pd.read_csv(csv_path, sep=None, engine="python")
        df.columns = [str(c).strip() for c in df.columns]
        return df
    except:
        return None

def _split(items, train_r, val_r, seed):
    rng = random.Random(seed)
    temp = items[:]
    rng.shuffle(temp)
    n = len(temp)
    n_tr = int(n * train_r)
    n_vl = int(n * val_r)
    return temp[:n_tr], temp[n_tr:n_tr+n_vl], temp[n_tr+n_vl:]

TRAIN_R, VAL_R, TEST_R, SEED = 0.72, 0.14, 0.14, 42

# Doc CSV tim File Path
by_class = {}
csv_files = sorted(Path(CSV_DIR).glob("*.csv"))
print(f"Tim thay {len(csv_files)} CSV files")

for csv_path in csv_files:
    df = _read_csv(csv_path)
    if df is None:
        continue
    lower_cols = {c.lower(): c for c in df.columns}
    file_col = None
    for cand in ["file path", "filepath", "path", "file_path"]:
        if cand in lower_cols:
            file_col = lower_cols[cand]
            break
    if file_col is None:
        print(f"  [SKIP] {csv_path.name} — khong co cot File Path")
        continue

    for _, row in df.iterrows():
        rel = str(row[file_col]).strip()
        parts = Path(rel).parts
        if len(parts) < 2:
            continue
        cls = parts[0]
        abs_path = str(Path(DATASET_ROOT) / rel)
        by_class.setdefault(cls, []).append(abs_path)

# Dedup
for cls in by_class:
    by_class[cls] = list(dict.fromkeys(by_class[cls]))

manifest = {"seed": SEED, "splits": {"train": TRAIN_R, "val": VAL_R, "test": TEST_R},
            "classes": sorted(by_class.keys()),
            "data": {"train": {}, "val": {}, "test": {}}}

print(f"\n{'Class':25s}  {'Total':>6s}  {'Train':>6s}  {'Val':>6s}  {'Test':>6s}")
print("-" * 58)
for cls, items in sorted(by_class.items()):
    tr, vl, te = _split(items, TRAIN_R, VAL_R, SEED)
    manifest["data"]["train"][cls] = tr
    manifest["data"]["val"][cls]   = vl
    manifest["data"]["test"][cls]  = te
    print(f"{cls:25s}  {len(items):6d}  {len(tr):6d}  {len(vl):6d}  {len(te):6d}")

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\nManifest luu tai: {MANIFEST_PATH}")

## Bước 2 — Extract Face Crops

- **32 frame/video** uniform sampling  
- **MTCNN** detect khuôn mặt (GPU)  
- Dataset đã ở local disk Kaggle → không bị chậm do mạng  
- Parallel: đọc N video song song trên CPU, MTCNN chạy GPU

In [ ]:
import cv2, time, threading, numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from facenet_pytorch import MTCNN as _MTCNN

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
MTCNN_BATCH  = 256    # tang len neu GPU con nho
READ_WORKERS = 8      # doc video song song tren CPU

print(f"Device: {DEVICE} | MTCNN batch: {MTCNN_BATCH} | read workers: {READ_WORKERS}")

# MTCNN + lock (GPU khong thread-safe)
detector = _MTCNN(
    margin=50, min_face_size=100,
    thresholds=[0.6, 0.7, 0.7], factor=0.7,
    post_process=False, select_largest=False, keep_all=True,
    device=DEVICE
)
_lock = threading.Lock()

# ── Helpers ───────────────────────────────────────────────────────────────────
def _uniform_idx(total, n):
    if total <= n: return list(range(total))
    return np.linspace(0, total-1, n).astype(int).tolist()

def _read_frames(video_path, indices):
    """Doc dung frame can (khong doc toan bo video)."""
    cap = cv2.VideoCapture(video_path)
    frames, prev = [], -1
    for idx in sorted(set(indices)):
        if idx > prev + 80:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        else:
            while int(cap.get(cv2.CAP_PROP_POS_FRAMES)) < idx:
                cap.grab()
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        prev = idx
    cap.release()
    return frames

def _square_crop(frame, box, margin=0.30):
    x1,y1,x2,y2 = [int(b) for b in box]
    side = int(max(x2-x1, y2-y1) * (1+margin))
    cx,cy = (x1+x2)/2, (y1+y2)/2
    sx1,sy1 = int(round(cx-side/2)), int(round(cy-side/2))
    pl=max(0,-sx1); pt=max(0,-sy1)
    pr=max(0,sx1+side-frame.shape[1]); pb=max(0,sy1+side-frame.shape[0])
    if pl or pt or pr or pb:
        frame = cv2.copyMakeBorder(frame,pt,pb,pl,pr,cv2.BORDER_REFLECT)
    return frame[sy1+pt:sy1+pt+side, sx1+pl:sx1+pl+side]

def process_one(video_path, out_dir, n_frames):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if total <= 0: return 0

    frames = _read_frames(video_path, _uniform_idx(total, n_frames))
    if not frames: return 0

    saved, stem = 0, Path(video_path).stem
    for i in range(0, len(frames), MTCNN_BATCH):
        batch = frames[i:i+MTCNN_BATCH]
        with _lock:
            boxes_list, _ = detector.detect(batch)
        for frame, boxes in zip(batch, boxes_list):
            if boxes is None: continue
            for box in boxes:
                face = _square_crop(frame, box)
                if face.shape[0] < 10 or face.shape[1] < 10: continue
                cv2.imwrite(
                    os.path.join(out_dir, f"{stem}-{time.time():.6f}.jpg"),
                    cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
                )
                saved += 1
    return saved

def extract_split(split, manifest_path, extracted_root, n_frames, n_workers=READ_WORKERS):
    with open(manifest_path) as f:
        data = json.load(f)["data"][split]

    total_faces = 0
    for cls_name, video_paths in sorted(data.items()):
        out_dir = os.path.join(extracted_root, split, cls_name)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n[{split}] {cls_name}: {len(video_paths)} videos")

        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futs = {ex.submit(process_one, vp, out_dir, n_frames): vp
                    for vp in video_paths}
            with tqdm(total=len(video_paths), desc=f"  {cls_name}") as pbar:
                for fut in as_completed(futs):
                    try:
                        total_faces += fut.result()
                    except Exception as e:
                        print(f"  Error {futs[fut]}: {e}")
                    pbar.update(1)

    print(f"[{split}] Tong mat: {total_faces}")
    return total_faces

print("Ham extract san sang")

In [ ]:
# Chay extract cho ca 3 splits
for split in ["train", "val", "test"]:
    t0 = time.time()
    extract_split(split, MANIFEST_PATH, EXTRACTED_ROOT, FRAMES_PER_VIDEO)
    print(f"[{split}] Xong: {(time.time()-t0)/60:.1f} phut\n")

print("EXTRACT HOAN TAT!")

In [ ]:
import os

username = os.environ.get("KAGGLE_USERNAME")

print("Kaggle username:", username)

In [ ]:
# Them cell nay NGAY SAU extract xong
# Output se thanh dataset vinh vien, khong bao gio mat

import json, subprocess, os

KAGGLE_USERNAME = "ten_kaggle_cua_ban"  # ← sua o day

meta = {
    "title": "FFPP C23 Extracted Faces 32f",
    "id": f"{KAGGLE_USERNAME}/ffpp-c23-extracted-32f",
    "licenses": [{"name": "other"}]
}
with open("/kaggle/working/extracted_faces/dataset-metadata.json", "w") as f:
    json.dump(meta, f)

# Can co /root/.kaggle/kaggle.json (API token)
# Lay tai: kaggle.com → Account → API → Create New Token
!kaggle datasets create -p /kaggle/working/extracted_faces --dir-mode zip
print("Da luu thanh Kaggle Dataset — khong bao gio mat!")

In [ ]:
# Thong ke so anh da extract
print(f"{'Split':6s}  {'Class':25s}  {'Anh':>8s}")
print("-" * 46)
total_imgs = 0
for split in ["train", "val", "test"]:
    split_dir = os.path.join(EXTRACTED_ROOT, split)
    if not os.path.exists(split_dir): continue
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".png"))])
        total_imgs += n
        print(f"{split:6s}  {cls:25s}  {n:8,}")

# Dung luong
import shutil
used = shutil.disk_usage("/kaggle/working")
print(f"\nTong so anh  : {total_imgs:,}")
print(f"Dung luong   : {used.used/1e9:.1f} GB / {used.total/1e9:.1f} GB")

## Bước 3 — Lưu thành Kaggle Dataset (để dùng lại)

Sau khi extract xong, lưu thành dataset mới để:
- Attach vào notebook training mà không cần extract lại
- Download về máy local